# RAG Ingestion Pipeline
## Petunjuk Teknis Budidaya Tanaman Sayuran

Pipeline ini membaca PDF, membuat chunk teks, menghasilkan embedding multilingual,
dan menyimpannya ke ChromaDB.

**Urutan eksekusi:**
1. Install dependencies
2. Upload PDF ke Colab
3. Ekstrak dan chunk teks
4. Build ChromaDB vector store
5. Validasi hasil ingestion
6. Export folder `chroma_db/` ke Google Drive

In [1]:
# ── CELL 1: Install dependencies (versi pinned untuk Colab) ──────────────────
!pip install -q -U pip
!pip install -q \
    chromadb==0.5.23 \
    sentence-transformers==2.7.0 \
    transformers==4.39.3 \
    tokenizers==0.15.2 \
    huggingface-hub==0.23.4 \
    pypdf==5.1.0 \
    pdfplumber==0.11.4 \
    langdetect==1.0.9

# Restart runtime setelah install (WAJIB sekali saja)
print("\n✅ Install selesai.")
print("⚠️  PENTING: Klik Runtime > Restart session, lalu lanjut ke Cell 2.")
print("   Jangan jalankan Cell 1 lagi setelah restart.")


✅ Install selesai.
⚠️  PENTING: Klik Runtime > Restart session, lalu lanjut ke Cell 2.
   Jangan jalankan Cell 1 lagi setelah restart.


In [2]:
# ── CELL 2: Upload PDF ────────────────────────────────────────────────────────
from google.colab import files

print("Upload file PDF petunjuk teknis budidaya sayuran:")
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print(f"File tersimpan: {PDF_PATH}")

Upload file PDF petunjuk teknis budidaya sayuran:


Saving Petunjuk-Teknis-Budidaya-Tanaman-Sayuran.pdf to Petunjuk-Teknis-Budidaya-Tanaman-Sayuran.pdf
File tersimpan: Petunjuk-Teknis-Budidaya-Tanaman-Sayuran.pdf


In [3]:
# ── CELL 3: Ekstrak teks dari PDF ────────────────────────────────────────────
import pdfplumber
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path: str) -> list[dict]:
    """
    Ekstrak teks per halaman dari PDF.
    Fallback ke pypdf jika pdfplumber gagal pada halaman tertentu.
    Mengembalikan list of dict: {page_num, text, char_count}
    """
    pages = []
    reader_backup = PdfReader(pdf_path)

    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if not text or len(text.strip()) < 20:
                text = reader_backup.pages[i].extract_text() or ""
            text = text.strip()
            if text:
                pages.append({
                    "page_num": i + 1,
                    "text": text,
                    "char_count": len(text)
                })
    return pages

raw_pages = extract_text_from_pdf(PDF_PATH)
total_chars = sum(p["char_count"] for p in raw_pages)
print(f"Total halaman berhasil diekstrak : {len(raw_pages)}")
print(f"Total karakter                   : {total_chars:,}")
print(f"\nContoh halaman 1 (200 karakter pertama):")
print(raw_pages[0]["text"][:200] if raw_pages else "[Tidak ada teks]")

Total halaman berhasil diekstrak : 142
Total karakter                   : 187,552

Contoh halaman 1 (200 karakter pertama):
Petunjuk Teknis
Budidaya Tanaman Sayuran
Oleh :
Wiwin Setiawati, Rini Murtiningsih,
Gina Aliya Sopha dan Tri Handayani
BALAI PENELITIAN TANAMAN SAYURAN
PUSAT PENELITIAN DAN PENGEMBANGAN HORTIKULTURA
B


In [4]:
# ── CELL 4: Chunking dengan overlap ──────────────────────────────────────────
from langdetect import detect

CHUNK_SIZE = 500       # karakter per chunk
CHUNK_OVERLAP = 80     # overlap antar chunk untuk konteks

def chunk_text(text: str, page_num: int, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[dict]:
    """
    Membagi teks menjadi chunk dengan sliding window overlap.
    Mencoba memotong di batas kalimat (.!?) untuk kualitas lebih baik.
    """
    chunks = []
    start = 0
    chunk_idx = 0

    while start < len(text):
        end = start + chunk_size
        segment = text[start:end]

        if end < len(text):
            # Cari batas kalimat terdekat
            for delim in [". ", "! ", "? ", "\n"]:
                boundary = segment.rfind(delim)
                if boundary > chunk_size // 2:
                    segment = segment[:boundary + 1]
                    break

        segment = segment.strip()
        if len(segment) > 50:
            try:
                lang = detect(segment)
            except Exception:
                lang = "id"

            chunks.append({
                "chunk_id": f"page{page_num}_chunk{chunk_idx}",
                "text": segment,
                "page_num": page_num,
                "chunk_idx": chunk_idx,
                "char_count": len(segment),
                "lang": lang
            })
            chunk_idx += 1

        start += len(segment) - overlap if len(segment) > overlap else len(segment)

    return chunks

all_chunks = []
for page in raw_pages:
    chunks = chunk_text(page["text"], page["page_num"])
    all_chunks.extend(chunks)

print(f"Total chunk dibuat  : {len(all_chunks)}")
print(f"Rata-rata per halaman: {len(all_chunks)/len(raw_pages):.1f} chunk")
print(f"\nContoh chunk pertama:")
print(f"  ID   : {all_chunks[0]['chunk_id']}")
print(f"  Lang : {all_chunks[0]['lang']}")
print(f"  Text : {all_chunks[0]['text'][:200]}")

Total chunk dibuat  : 722
Rata-rata per halaman: 5.1 chunk

Contoh chunk pertama:
  ID   : page2_chunk0
  Lang : id
  Text : Petunjuk Teknis
Budidaya Tanaman Sayuran
Oleh :
Wiwin Setiawati, Rini Murtiningsih,
Gina Aliya Sopha dan Tri Handayani
BALAI PENELITIAN TANAMAN SAYURAN
PUSAT PENELITIAN DAN PENGEMBANGAN HORTIKULTURA
B


In [5]:
# ── CELL 5: Build ChromaDB dengan multilingual embedding ──────────────────────
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer
import os

CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "sayuran_kb"
EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"  # 118MB, mendukung ID+EN

os.makedirs(CHROMA_DIR, exist_ok=True)

# Load model langsung — BYPASS chromadb wrapper yang bermasalah di Colab
print(f"Memuat embedding model: {EMBED_MODEL}")
st_model = SentenceTransformer(EMBED_MODEL)
print("Model berhasil dimuat!")

# Custom embedding function yang tidak bergantung chromadb.utils
class DirectEmbeddingFunction:
    """Wrapper langsung SentenceTransformer untuk ChromaDB."""
    def __init__(self, model):
        self.model = model

    def __call__(self, input: list[str]) -> list[list[float]]:
        embeddings = self.model.encode(input, normalize_embeddings=True)
        return embeddings.tolist()

embedding_fn = DirectEmbeddingFunction(st_model)

client = chromadb.PersistentClient(path=CHROMA_DIR)

# Hapus collection lama kalau ada (untuk re-ingest bersih)
existing = [c.name for c in client.list_collections()]
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)
    print(f"Collection lama '{COLLECTION_NAME}' dihapus.")

collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

# Batch insert
BATCH_SIZE = 50
total_inserted = 0

for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i:i + BATCH_SIZE]
    collection.add(
        ids=[c["chunk_id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[{
            "page_num": c["page_num"],
            "chunk_idx": c["chunk_idx"],
            "char_count": c["char_count"],
            "lang": c["lang"]
        } for c in batch]
    )
    total_inserted += len(batch)
    print(f"  Batch {i//BATCH_SIZE + 1}: {total_inserted}/{len(all_chunks)} chunk tersimpan")

print(f"\nSelesai! Total {total_inserted} chunk di collection '{COLLECTION_NAME}'.")
print(f"ChromaDB tersimpan di: {CHROMA_DIR}/")

Memuat embedding model: paraphrase-multilingual-MiniLM-L12-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model berhasil dimuat!


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


  Batch 1: 50/722 chunk tersimpan
  Batch 2: 100/722 chunk tersimpan
  Batch 3: 150/722 chunk tersimpan
  Batch 4: 200/722 chunk tersimpan
  Batch 5: 250/722 chunk tersimpan
  Batch 6: 300/722 chunk tersimpan
  Batch 7: 350/722 chunk tersimpan
  Batch 8: 400/722 chunk tersimpan
  Batch 9: 450/722 chunk tersimpan
  Batch 10: 500/722 chunk tersimpan
  Batch 11: 550/722 chunk tersimpan
  Batch 12: 600/722 chunk tersimpan
  Batch 13: 650/722 chunk tersimpan
  Batch 14: 700/722 chunk tersimpan
  Batch 15: 722/722 chunk tersimpan

Selesai! Total 722 chunk di collection 'sayuran_kb'.
ChromaDB tersimpan di: chroma_db/


In [6]:
# ── CELL 6: Validasi — query test retrieval ───────────────────────────────────

def test_retrieval(query: str, n_results: int = 3):
    """Test retrieval dan tampilkan skor relevansi."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    print(f"Query: '{query}'")
    print("─" * 60)
    for j, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )):
        relevance_score = round((1 - dist) * 100, 1)
        print(f"  [{j+1}] Halaman {meta['page_num']} | Relevansi: {relevance_score}%")
        print(f"       {doc[:200]}...")
        print()

# Test beberapa query
test_queries = [
    "cara menanam cabai",
    "penyakit tanaman tomat",
    "kebutuhan air bayam",
    "pupuk organik sayuran",
    "hama ulat pada kangkung"
]

for q in test_queries:
    test_retrieval(q, n_results=2)
    print("=" * 60)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: 'cara menanam cabai'
────────────────────────────────────────────────────────────
  [1] Halaman 75 | Relevansi: 59.5%
       tanaman dan pengasapan
menggunakan belerang.
Balai Penelitian Tanaman Sayuran 68...

  [2] Halaman 38 | Relevansi: 57.8%
       Petunjuk Teknis Prima Tani W. Setiawati, R. Murtiningsih,
G.A. Sopha, dan T. Handayati :
Budidaya Tanaman Sayuran
berhubungan dengan tempat tumbuh tanaman cabai (sawah atau
tegalan). Tanaman cabai yan...

Query: 'penyakit tanaman tomat'
────────────────────────────────────────────────────────────
  [1] Halaman 116 | Relevansi: 70.3%
       a semiclausum)
- Tumpangsari petsai– tomat.
Balai Penelitian Tanaman Sayuran 109...

  [2] Halaman 129 | Relevansi: 67.2%
       ntuk pertumbuhan tomat antara 21-240C. Waktu
Balai Penelitian Tanaman Sayuran 122...

Query: 'kebutuhan air bayam'
────────────────────────────────────────────────────────────
  [1] Halaman 26 | Relevansi: 71.3%
       a
dengan pemupukan yang teratur.
Kebutuhan air unt

In [8]:
# ── CELL 7: Simpan ke Google Drive (opsional) ─────────────────────────────────
import shutil
from google.colab import drive

SAVE_TO_DRIVE = True  # Set False kalau tidak mau upload ke Drive

if SAVE_TO_DRIVE:
    drive.mount("/content/drive")
    DRIVE_PATH = "/content/drive/MyDrive/sayuran-rag/chroma_db"
    os.makedirs(DRIVE_PATH, exist_ok=True)
    shutil.copytree(CHROMA_DIR, DRIVE_PATH, dirs_exist_ok=True)
    print(f"ChromaDB tersimpan ke Google Drive: {DRIVE_PATH}")
    print("Download folder chroma_db/ dari Drive untuk dipakai di Streamlit Cloud.")
else:
    print("Melewati penyimpanan ke Drive.")
    print("Download manual: Files > chroma_db/ > klik kanan > Download")

Mounted at /content/drive
ChromaDB tersimpan ke Google Drive: /content/drive/MyDrive/sayuran-rag/chroma_db
Download folder chroma_db/ dari Drive untuk dipakai di Streamlit Cloud.


In [9]:
# ── CELL 8: Ringkasan ingestion ───────────────────────────────────────────────

lang_counts = {}
for c in all_chunks:
    lang_counts[c["lang"]] = lang_counts.get(c["lang"], 0) + 1

print("=" * 50)
print("  RINGKASAN INGESTION")
print("=" * 50)
print(f"  PDF          : {PDF_PATH}")
print(f"  Halaman      : {len(raw_pages)}")
print(f"  Chunk        : {len(all_chunks)}")
print(f"  Chunk size   : {CHUNK_SIZE} karakter (overlap {CHUNK_OVERLAP})")
print(f"  Embedding    : {EMBED_MODEL}")
print(f"  Vector store : {CHROMA_DIR}/")
print(f"  Collection   : {COLLECTION_NAME}")
print(f"  Distribusi bahasa:")
for lang, count in sorted(lang_counts.items(), key=lambda x: -x[1]):
    pct = count / len(all_chunks) * 100
    print(f"    {lang:6s}: {count:4d} chunk ({pct:.1f}%)")
print("=" * 50)
print("  Ingestion SELESAI. Siap untuk app.py")
print("=" * 50)

  RINGKASAN INGESTION
  PDF          : Petunjuk-Teknis-Budidaya-Tanaman-Sayuran.pdf
  Halaman      : 142
  Chunk        : 722
  Chunk size   : 500 karakter (overlap 80)
  Embedding    : paraphrase-multilingual-MiniLM-L12-v2
  Vector store : chroma_db/
  Collection   : sayuran_kb
  Distribusi bahasa:
    id    :  720 chunk (99.7%)
    tl    :    1 chunk (0.1%)
    en    :    1 chunk (0.1%)
  Ingestion SELESAI. Siap untuk app.py
